# Scraping a news website for headlines

This code scrapes information from Le Monde's website (https://www.lemonde.fr/en/) to create a csv with each article on the homepages' title, subhead, article URL, whether it's premium or not, byline, article type and image URL. 

## Importing the necessary packages and data

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Getting the homepage data and saving it in a variable called doc

In [2]:
response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser')

## Searching the webpage data for the elements I need

Based on using the inspect tool on the website, I found the element div.article.article--headlines is being used to display several articles on the page. But that doesn't look like it will work for everything as different articles display on the page different. Thus I started by at any element with the article class.

In [3]:
stories = doc.find_all(class_="article")
stories

[<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96644"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96644" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/le-monde-africa/article/2026/06/27/one-year-after-replacing-wagner-russia-s-africa-corps-struggles-in-mali_6754922_124.html"> <p class="article__title-label">One year after replacing Wagner, Russia's Africa Corps struggles in Mali</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Damaged infrastructure inside the former Africa Corps barracks at Camp 2 in Kidal (Mali), May 11, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/23/0/0/5760/3840/400/266/75/0/d4f5b9e_upload-1-ssoa7nvute5u-000-b2hr4n2.jpg" srcset=" https://img.lemde.fr/2026/06/

#### First attempt at exploring the data using Soma's example code from class

In [4]:
# Starting off without ANY rows
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        print("Headline not found!")
        continue
        
    try:
        # Find me a media__link OR a reel_link
        row['href'] = story.select_one('a')['href']
    except:
        print("Couldn't find a link")

    try:
        row['tag'] = story.select_one('.ewmvhB').text.strip()
    except:
        print("Couldn't find a tag!")

    try:
        row['tag'] = story.find(attrs={'data-testid': 'card-description'}).text.strip()
    except:
        print("Couldn't find a summary")

    print(row)
    # When we're done adding info to our row, we're going to add it into our list
    # of rows
    rows.append(row)

----
Couldn't find a tag!
Couldn't find a summary
{'title': "One year after replacing Wagner, Russia's Africa Corps struggles in Mali", 'href': 'https://www.lemonde.fr/en/le-monde-africa/article/2026/06/27/one-year-after-replacing-wagner-russia-s-africa-corps-struggles-in-mali_6754922_124.html'}
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!


That did not work at all. So I will start exploring the data on my own.

#### Exploring the first 3 and last 3 stories myself just to get a sense of what elements are in it

In [5]:
for story in stories[:3]:
    print("----")
    print(story)

print(" ----- last 3 ----- ")

for story in stories[(len(stories)-3):]:
    print("----")
    print(story)

----
<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-96644"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-96644" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/le-monde-africa/article/2026/06/27/one-year-after-replacing-wagner-russia-s-africa-corps-struggles-in-mali_6754922_124.html"> <p class="article__title-label">One year after replacing Wagner, Russia's Africa Corps struggles in Mali</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="Damaged infrastructure inside the former Africa Corps barracks at Camp 2 in Kidal (Mali), May 11, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/06/23/0/0/5760/3840/400/266/75/0/d4f5b9e_upload-1-ssoa7nvute5u-000-b2hr4n2.jpg" srcset=" https://img.lemde.fr/2026

## Building a scraper to capture the information I need

We are looking for...<br>
headline = article__title class <br>
subtitle = article__desc class (only exists for some headlines) <br>
article url = lmd-link-clickarea__link class <br>
image url = src attribute withing the image tag <br>
bylines = article__byline class (but most aren't shown on the homepage) another option I have seen is article__author-name <br>
whether it is premium or not = sr-only class <br>
article type = article__type class (doesn't exist for all stories) <br>

This code uses the above information in a for loop to build a data set with the information we need. I am printing the information for each step to make sure it is working as planned.

In [6]:
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        # This gets the headlines generally but will need to be cleaned up using pandas
        row["headline"] = (story.select_one(".article__title").get_text())
        print(row["headline"])
    except:
        print("Headline not found!")
        continue
        
    try:
        # finding the subtitles
        row["subtitle"] = (story.select_one(".article__desc").get_text())
        print(row["subtitle"])
    except:
        print("Couldn't find a subtitle")

    # The .lmd-link-clickarea__link is not getting the article urls for everything unfortunately
    # so I have added one more attempt to catch stragglers before instructing the loop to give up
    try:
        row["article_url"] = story.select_one(".lmd-link-clickarea__link")["href"]
        print(row["article_url"])
    except:
        try:
            row["article_url"] = story.select_one("a")["href"]
            print(row["article_url"])
        except:
            print("Couldn't find an article url")

    try:
        row["image_url"] = story.select_one("img")["src"]
        print(row["image_url"])
    except:
        print("Couldn't find an image_url")
    
    # Bylines appear under different names and many of the articles do not have bylines at all on the homepage
    try:
        row["byline"] = (story.select_one(".article__byline").get_text())
        print(f"byline {row["byline"]}")
    except:
        try:
            row["byline"] = (story.select_one(".article__author-name").get_text())
            print(f"author name {row["byline"]}")
        except:
            print("Couldn't find a byline")

    try:
        if story.select_one(".sr-only") != None:
            row["paywall_status"] = "subscriber only"
            print(row["paywall_status"])
    except:
        print("Couldn't find subscriber status")

    try:
        row["article_type"] = (story.select_one(".article__type").get_text())
        print(row["article_type"])
    except:
        print("Couldn't find article type")
        
    # # When we're done adding info to our row, we're going to add it into our list    
    print(row)
    rows.append(row)

----
 One year after replacing Wagner, Russia's Africa Corps struggles in Mali  
'Mali: One year of Africa Corps' (1/2). Attacks by jihadists and Tuareg separatists in several regions of the country shook the ruling junta in late April.
https://www.lemonde.fr/en/le-monde-africa/article/2026/06/27/one-year-after-replacing-wagner-russia-s-africa-corps-struggles-in-mali_6754922_124.html
https://img.lemde.fr/2026/06/23/0/0/5760/3840/400/266/75/0/d4f5b9e_upload-1-ssoa7nvute5u-000-b2hr4n2.jpg
Couldn't find a byline
subscriber only
Couldn't find article type
{'headline': " One year after replacing Wagner, Russia's Africa Corps struggles in Mali  ", 'subtitle': "'Mali: One year of Africa Corps' (1/2). Attacks by jihadists and Tuareg separatists in several regions of the country shook the ruling junta in late April.", 'article_url': 'https://www.lemonde.fr/en/le-monde-africa/article/2026/06/27/one-year-after-replacing-wagner-russia-s-africa-corps-struggles-in-mali_6754922_124.html', 'image_url'

## Exporting the results

Saving it as a data frame and having a quick glance

In [7]:
df = pd.DataFrame(rows)
df

,headline,subtitle,article_url,image_url,paywall_status,byline,article_type
0,"One year after replacing Wagner, Russia's Afr...",'Mali: One year of Africa Corps' (1/2). Attack...,https://www.lemonde.fr/en/le-monde-africa/arti...,https://img.lemde.fr/2026/06/23/0/0/5760/3840/...,subscriber only,NaN,NaN
1,Heatwave fuels frustration in Paris's northern...,NaN,https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/06/25/0/0/2835/1890/...,subscriber only,NaN,NaN
2,'Venezuela is an earthquake-prone country. 80%...,NaN,https://www.lemonde.fr/en/environment/article/...,https://img.lemde.fr/2026/06/25/0/0/5752/3835/...,subscriber only,NaN,NaN
3,Parents outraged by lack of school guidelines ...,"As classroom conditions worsen, parents have t...",https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/06/23/0/0/4252/2834/...,subscriber only,NaN,NaN
4,250 years of American independence: When John ...,"'America 250' (5/13). John Adams, the second p...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/26/379/0/1167/778...,subscriber only,NaN,NaN
5,Mansplashing: Why do men splash everyone at th...,"The behavior of the so-called 'pool man,' who ...",https://www.lemonde.fr/en/france/article/2026/...,https://img.lemde.fr/2026/06/24/692/0/2553/170...,subscriber only,NaN,NaN
6,"La Maison du Bassin, a secluded wooden retreat...",In the south of the Cap Ferret peninsula on th...,https://www.lemonde.fr/en/lifestyle/article/20...,https://img.lemde.fr/2026/06/10/311/0/1035/690...,subscriber only,Hélène Brunet-Rivaillon,Feature
7,"Sleeping in a cellar, waiting at a funeral hom...",NaN,https://www.lemonde.fr/en/environment/article/...,https://img.lemde.fr/2026/06/23/0/0/5049/3366/...,subscriber only,NaN,NaN
8,Russia-annexed Crimea declares 'emergency' ami...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/26/0/28/8044/5363...,NaN,NaN,NaN
9,Paris Pride march postponed over heatwave,NaN,https://www.lemonde.fr/en/lgbtq/article/2026/0...,https://img.lemde.fr/2026/06/26/0/0/5568/3712/...,NaN,NaN,NaN


Exporting the data as a csv

In [8]:
df.to_csv("le_monde_daily_news.csv")

## Making the CSV auto-updating

I have created a github action to autoupdate this code, using Jonathan Soma's tutorial
https://www.youtube.com/watch?v=QNKxzkNpsko.

For future reference:
Step 1: Make your working files into a github repository
Step 2: Create a github action called update.yml using Soma's code 
Step 3: Make sure that you use a Cron code for the frequency of update that you want and that the file referenced is the correct one. Try to run the action and trouble shoot any error by checking at which point the code stopped being able to run.
